# Notebook 03 — Extraction and Precision



**Duration:** ~30 minutes



In notebooks 01 and 02 you connected to an LLM and explored how it behaves. Now we give it real data for the first time.



The dataset is the **Privacy Act 2020** — a New Zealand Act of Parliament published as XML on the government's legislation website. We will fetch it live, parse it into sections, and then try two different methods for extracting structured information from those sections:



1. A **rule-based method** (regular expressions) — deterministic, precise, but rigid

2. An **LLM-based method** — flexible, but probabilistic and sometimes wrong



By the end of this notebook you will have measured the gap between these two approaches on real data — and started thinking about what that means for research at scale.

## Setup

Run this cell first. If you stored your Groq API key in Colab Secrets (notebook 01), it will load automatically.

In [ ]:
# ============================================================
# SETUP CELL — Run this once at the start of every notebook
# ============================================================

!pip install groq requests lxml Pillow
import os, json, base64, requests, io
from groq import Groq
from lxml import etree
from PIL import Image
from IPython.display import Image as IPImage, display

# Load API key from Colab Secrets (set up in notebook 01)
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    os.environ["GROQ_API_KEY"] = "paste_your_key_here"   # <-- fallback: paste key here
    print("Could not load from Secrets. Paste your key in the line above.")

client = Groq(api_key=os.environ["GROQ_API_KEY"])
TEXT_MODEL = "llama-3.3-70b-versatile"
VISION_MODEL = "meta-llama/llama-4-maverick-17b-128e-instruct"

print("Setup complete.")

## Load the dataset

The New Zealand government publishes every Act of Parliament as an **XML** file — a structured text format that organises content using labelled tags.

For example, each section of an Act is wrapped in a `<prov>` tag (short for "provision"). Inside that, `<label>` holds the section number and the text content is nested within `<prov.body>`.

We will fetch the Privacy Act 2020 directly from the legislation website, then parse it into a list of sections we can loop over.

Run the cell below. It downloads the Act and extracts every section.

In [ ]:
import re

# Fetch the Privacy Act 2020 XML
# Cached in this repo because legislation.govt.nz blocks requests from cloud IPs (e.g. Colab).
# Original source: https://legislation.govt.nz/act/public/2020/31/en/latest.xml
url = "https://raw.githubusercontent.com/UoA-eResearch/embedding-llms-qualitative-data-workshop/main/data/privacy-act-2020.xml"
response = requests.get(url, timeout=30)

if response.status_code != 200 or len(response.content) < 100:
    raise RuntimeError(
        f"Could not fetch the Act (status {response.status_code}, "
        f"{len(response.content)} bytes). Check your internet connection."
    )

root = etree.fromstring(response.content)

# Parse each <prov> element into a dictionary with label, heading, and text
sections = []
for prov in root.findall(".//prov"):
    label_el = prov.find(".//label")
    heading_el = prov.find(".//heading")
    body_el = prov.find(".//prov.body")

    label = label_el.text.strip() if label_el is not None and label_el.text else "?"
    heading = heading_el.text.strip() if heading_el is not None and heading_el.text else ""
    text = "".join(body_el.itertext()).strip() if body_el is not None else ""

    sections.append({"label": label, "heading": heading, "text": text})

print(f"Loaded {len(sections)} sections from the Privacy Act 2020.")
print()
print("First three sections:")
for s in sections[:3]:
    print(f"  Section {s['label']}: {s['heading']}")

You should see around 240 sections. Each one has:
- A **label** (the section number, e.g. "3")
- A **heading** (a short title, e.g. "Purpose of this Act")
- The **text** (the full body of that section)

This is Step 1 of the research workflow from notebook 01 — **ingestion**. Raw data, parsed into a usable format.

Let's look at one section in full so you can see what the text looks like.

In [ ]:
# Pick section 86 ("Power to summon persons") — a mid-length section that references other Acts
example = next(s for s in sections if s["label"] == "86")

print(f"Section {example['label']}: {example['heading']}")
print("=" * 60)
print(example["text"])

Notice the references to other Acts embedded in the text — things like "Crimes Act 1961" and "Criminal Procedure Act 2011". These cross-references tell you which other laws this section connects to.

Extracting these cross-references is a real research task. It is also a perfect test case for comparing rule-based and LLM-based methods.

## The extraction dilemma

> **Research context:** The published study on New Zealand's Mental Health Act (Ardekani et al., 2026) faced this exact problem: how do you extract cross-references from legislative text at scale? They tested both a rule-based engine and an LLM. The rule-based method achieved **96% precision**. The LLM achieved **86%**. The LLM was more flexible — it could find implicit references that no rule anticipated — but it was less consistent. The authors called this the **extraction dilemma**.

We are now going to reproduce this dilemma on a smaller scale. You will:

1. Use a **regular expression** (regex) to find Act references in the text
2. Use the **LLM** to find Act references in the same text
3. Compare what each method found

A **regular expression** is a pattern that describes what you are looking for in text. For example, the pattern `[A-Z]...Act \d{4}` says: "find text that starts with a capital letter, contains the word Act, and ends with a four-digit year." It is deterministic — it finds exactly what the pattern describes, nothing more, nothing less.

## Exercise a — Citation detection: regex vs LLM

We will work with a sample of five sections that contain cross-references to other Acts. This keeps the exercise manageable and lets us verify results by hand.

In [ ]:
# Select five sections that reference other Acts
sample_labels = ["13", "86", "111", "138", "181"]
sample = [s for s in sections if s["label"] in sample_labels]

print(f"Sample: {len(sample)} sections")
for s in sample:
    print(f"  Section {s['label']}: {s['heading']} ({len(s['text'])} characters)")

### Step 1: Regex extraction

The regex pattern below looks for text that:
- Starts with a capital letter
- Contains one or more words
- Includes the word "Act"
- Ends with a four-digit year

It will catch references like "Crimes Act 1961" and "Official Information Act 1982".

In [ ]:
# Regex pattern for NZ Act references
pattern = r'[A-Z][a-zA-Z\s]+Act\s+\d{4}'

# Run regex on each section in the sample
regex_results = {}
for s in sample:
    matches = re.findall(pattern, s["text"])
    # Clean up: strip whitespace from each match, remove duplicates
    unique = sorted(set(m.strip() for m in matches))
    regex_results[s["label"]] = unique

# Print what the regex found
print("Regex results:")
print("=" * 60)
all_regex = set()
for label, acts in regex_results.items():
    print(f"\nSection {label}:")
    if acts:
        for a in acts:
            print(f"  - {a}")
            all_regex.add(a)
    else:
        print("  (none found)")

print(f"\nTotal unique Acts found by regex: {len(all_regex)}")

Look at the results. Some look correct (e.g. "Crimes Act 1961"). Others may look odd — the regex can accidentally capture extra words before "Act" because the pattern is greedy. This is a known limitation of regex: it finds exactly what the pattern describes, which is not always what you meant.

Now let's see what the LLM finds.

### Step 2: LLM extraction

We send the same sections to the LLM and ask it to list all NZ Acts referenced in each one. The LLM reads the text and uses its understanding of language — not a pattern — to identify references.

In [ ]:
# Send each section to the LLM and ask it to extract Act references
llm_results = {}

for s in sample:
    response = client.chat.completions.create(
        model=TEXT_MODEL,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": "You are a legal research assistant. You extract references to New Zealand Acts of Parliament from legislative text."
            },
            {
                "role": "user",
                "content": f"""List all New Zealand Acts of Parliament cited or referenced in the following text.
Return ONLY a JSON array of Act names, e.g. ["Crimes Act 1961", "Privacy Act 2020"].
If no Acts are referenced, return an empty array [].

Text:
{s['text']}"""
            }
        ]
    )

    # Parse the JSON response
    reply = response.choices[0].message.content.strip()
    try:
        acts = json.loads(reply)
    except json.JSONDecodeError:
        # If the model didn't return clean JSON, show the raw reply
        print(f"  Section {s['label']}: could not parse JSON — raw reply: {reply}")
        acts = []

    llm_results[s["label"]] = sorted(set(acts))

# Print what the LLM found
print("LLM results:")
print("=" * 60)
all_llm = set()
for label, acts in llm_results.items():
    print(f"\nSection {label}:")
    if acts:
        for a in acts:
            print(f"  - {a}")
            all_llm.add(a)
    else:
        print("  (none found)")

print(f"\nTotal unique Acts found by LLM: {len(all_llm)}")

### Step 3: Compare the two methods

Now we put the results side by side. For each Act reference, we check:
- **Both agree** — found by regex AND the LLM (high confidence)
- **LLM only** — the LLM found something the regex missed. Could be a genuine implicit reference, or could be a hallucination.
- **Regex only** — the regex matched something the LLM did not mention. Could be noise from the pattern, or the LLM missed it.

In [ ]:
# Compare: what did both methods find, and where did they disagree?
both = all_regex & all_llm
regex_only = all_regex - all_llm
llm_only = all_llm - all_regex

print(f"Regex found {len(all_regex)} unique Acts.")
print(f"LLM found {len(all_llm)} unique Acts.")
print(f"They agreed on {len(both)}.")
print()

if both:
    print("Both agree on:")
    for a in sorted(both):
        print(f"  - {a}")

if llm_only:
    print(f"\nLLM only ({len(llm_only)}):")
    for a in sorted(llm_only):
        print(f"  - {a}")

if regex_only:
    print(f"\nRegex only ({len(regex_only)}):")
    for a in sorted(regex_only):
        print(f"  - {a}")

### Step 4: Manual spot-check

The comparison gives you three categories. But which method is *right* when they disagree?

The only way to know is to check the source text yourself. This is the **human spot-check** — a targeted manual review of disputed cases.

The cell below prints the disputed references alongside the section text where they were (or were not) found. Read the text and decide: is the reference real?

In [ ]:
# Spot-check: for each disputed reference, show the section text so you can verify
disputed = (llm_only | regex_only)

if not disputed:
    print("No disputed references — both methods fully agree!")
else:
    print(f"{len(disputed)} disputed reference(s) to check:")
    print("=" * 60)

    for ref in sorted(disputed):
        source = "LLM only" if ref in llm_only else "Regex only"
        print(f"\n'{ref}' — found by: {source}")
        print("-" * 60)

        # Find which section(s) this appeared in
        for s in sample:
            in_regex = ref in regex_results.get(s["label"], [])
            in_llm = ref in llm_results.get(s["label"], [])
            if in_regex or in_llm:
                print(f"Section {s['label']}: {s['heading']}")
                # Show a snippet of the section text around the reference (if present)
                idx = s["text"].lower().find(ref.lower()[:20])
                if idx >= 0:
                    start = max(0, idx - 80)
                    end = min(len(s["text"]), idx + len(ref) + 80)
                    print(f"  ...{s['text'][start:end]}...")
                else:
                    print(f"  (reference text not found literally in section — may be paraphrased)")

    print("\n" + "=" * 60)
    print("For each disputed reference, ask yourself:")
    print("  1. Is this a real Act that exists?")
    print("  2. Is it actually referenced in the text, or was it invented?")
    print("  3. If the regex caught it oddly, is the Act name still recognisable?")

### What you just did

You reproduced — on a small scale — what the paper measured computationally:

| Method | Strength | Weakness |
|--------|----------|----------|
| Regex | Deterministic — same result every time. Fast. | Rigid — only finds what the pattern describes. Can capture noise. |
| LLM | Flexible — understands context, can find implicit references. | Probabilistic — may hallucinate references or miss some. |

> "You have done manually what the paper measured computationally. At 5 sections it is manageable. At 10,000 you need a protocol."

## Exercise b — How would you validate this at scale?

You just checked 5 sections by hand. But the Privacy Act has 240 sections. Other corpora might have thousands. You cannot read every one.

This is the **validation problem**. Researchers have developed several approaches:

| Approach | How it works | When to use it |
|----------|-------------|----------------|
| **Consistency check** | Run the same extraction twice and compare. If results differ, the method is unreliable. | First pass — quick sanity check (you did this in notebook 02 with temperature) |
| **Human spot-check** | Manually verify a random sample (e.g. 10% of outputs). | Second pass — targeted verification (you just did this above) |
| **Inter-rater agreement** | You and a colleague independently review the same sample. If you agree, the method is likely sound. | Formal validation for publication |
| **Cross-method comparison** | Compare two independent methods on the same data (e.g. regex vs LLM). Agreement builds confidence. | When you have access to more than one extraction approach |

The key references behind these approaches:

- **Ardekani et al. (2026)** measured LLM precision at 86% vs 96% for rule-based extraction on NZ legislative networks — the same comparison you just did.
- **Tai et al. (2024)** propose spot-checking 10% of LLM-coded outputs as a minimum standard for qualitative research using deductive coding.
- **Scaling approaches** (EPJ Data Science, Springer 2025) discuss how to maintain validation rigour as corpus size grows.

You do not need to memorise these. The point is: validation is not optional. It is part of the method.

### Discussion

Think about your own research for a moment:

1. If you used an LLM to code your data, which validation approach would you choose? Why?
2. What would you write in your methods section about the precision gap?
3. At what point would you trust the LLM's output enough to stop checking?

There is no single right answer. The paper's authors chose to use the rule-based method for their final results because precision mattered more than flexibility. For your research, the right choice depends on what you are doing and what your error tolerance is.

## Bonus: Run regex on the full Act

The LLM exercise used a 5-section sample to stay within API limits. But regex is fast — it can scan the entire Act in under a second. Run the cell below to see every Act referenced across all 240 sections.

In [ ]:
# Run regex across ALL sections
all_acts_full = set()
for s in sections:
    matches = re.findall(pattern, s["text"])
    for m in matches:
        all_acts_full.add(m.strip())

print(f"Regex found {len(all_acts_full)} unique Act references across all {len(sections)} sections.")
print()
for a in sorted(all_acts_full):
    print(f"  - {a}")

Some of these are clean ("Crimes Act 1961"). Others are noisy — the regex captured extra words before "Act" because the pattern is broad. A real research pipeline would add a cleaning step here. The LLM does that cleaning implicitly, which is part of its appeal — and part of the trade-off.

## What you accomplished

In this notebook you:

- Fetched and parsed real legislative data from the NZ government website (**Step 1: Ingestion**)
- Extracted cross-references using a rule-based method (regex) and an LLM (**Step 2: Extraction**)
- Compared the two methods and measured where they agree and disagree
- Manually verified disputed cases — your first **human spot-check**
- Learned four named validation approaches: consistency check, human spot-check, inter-rater agreement, and cross-method comparison

The key insight: extraction is not just about finding information. It is about knowing how much you can trust what you found. Both methods have blind spots. The question is whether those blind spots matter for your research question.

**Next up:** Notebook 04 moves from extraction to analysis. Instead of pulling out cross-references, you will ask the LLM to generate themes from the same corpus — and then test whether those themes hold up under scrutiny.